# Q5 — PCA and t-SNE Dimensionality Reduction

**Adaptation note:** the lab prompt names MNIST, but per lab instructions every
question must use the BSDS500-derived dataset only. PCA/t-SNE serve the same
purpose here — projecting a multi-dimensional feature space (colour + gradient +
texture + position) down to 2D to see whether the two classes (edge vs non-edge)
form separable clusters.

## Setup

This notebook expects to be run from the project root, alongside a `common.py`
(or with the helper cell below), and with the following layout already in place:

```
project_root/
├── archive/                # BSDS500 images + ground_truth
├── data/bsds_features.csv  # produced by the feature-extraction notebook
├── results/figures/
└── results/metrics/
```

If you don't have a `common.py` file in this directory, run the cell below first —
it defines the same `load_split` / `save_metrics` helpers used across all seven
questions so this notebook is self-contained.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
DATA_CSV = ROOT / "data" / "bsds_features.csv"
FIG_DIR = ROOT / "results" / "figures"
METRIC_DIR = ROOT / "results" / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["R", "G", "B", "gray", "grad_mag", "grad_dir",
                "laplacian", "local_std", "x_norm", "y_norm"]
LABEL_COL = "is_edge"
RANDOM_STATE = 42


def load_split(test_size=0.2, scale=True):
    df = pd.read_csv(DATA_CSV)
    X = df[FEATURE_COLS].values.astype(np.float64)
    y = df[LABEL_COL].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )

    if scale:
        mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)
        sigma[sigma == 0] = 1.0
        X_train = (X_train - mu) / sigma
        X_test = (X_test - mu) / sigma

    return X_train, X_test, y_train, y_test


def save_metrics(name, d):
    path = METRIC_DIR / f"{name}.json"
    with open(path, "w") as f:
        json.dump(d, f, indent=2, default=float)
    print(f"saved metrics -> {path}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
TSNE_SAMPLE_SIZE = 4000  # subsample for tractable t-SNE runtime

## Load and standardize the full dataset

In [ ]:
df = pd.read_csv(DATA_CSV)
X = df[FEATURE_COLS].values.astype(np.float64)
y = df[LABEL_COL].values.astype(int)

mu, sigma = X.mean(axis=0), X.std(axis=0)
sigma[sigma == 0] = 1.0
Xs = (X - mu) / sigma

## PCA (full dataset)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(Xs)
explained = pca.explained_variance_ratio_
print(f"PCA explained variance ratio: {explained} (sum={explained.sum():.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for label, name, color in [(0, "non-edge", "#4C72B0"), (1, "edge", "#DD8452")]:
    mask = y == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=4, alpha=0.35, c=color, label=name)
ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}% var)")
ax.set_title("PCA projection to 2D (full dataset)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(FIG_DIR / "q5_pca_2d.png", bbox_inches="tight")
plt.show()

## PCA loadings
Which original features drive PC1/PC2?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
comp = pca.components_
x = np.arange(len(FEATURE_COLS))
width = 0.35
ax.bar(x - width/2, comp[0], width, label="PC1")
ax.bar(x + width/2, comp[1], width, label="PC2")
ax.set_xticks(x); ax.set_xticklabels(FEATURE_COLS, rotation=45, ha="right")
ax.set_ylabel("loading")
ax.set_title("PCA component loadings")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "q5_pca_loadings.png", bbox_inches="tight")
plt.show()

## t-SNE (subsampled for speed)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(Xs), size=min(TSNE_SAMPLE_SIZE, len(Xs)), replace=False)
X_sub, y_sub = Xs[idx], y[idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, init="pca")
X_tsne = tsne.fit_transform(X_sub)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for label, name, color in [(0, "non-edge", "#4C72B0"), (1, "edge", "#DD8452")]:
    mask = y_sub == label
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=6, alpha=0.5, c=color, label=name)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title(f"t-SNE projection to 2D (subsample n={len(idx)})")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(FIG_DIR / "q5_tsne_2d.png", bbox_inches="tight")
plt.show()

## Parameter modification: sensitivity to perplexity

In [ ]:
sens_idx = idx[:1500]
X_sens, y_sens = Xs[sens_idx], y[sens_idx]
perplexities = [5, 15, 30, 50, 100]
sil_scores = []
for perp in perplexities:
    t = TSNE(n_components=2, perplexity=perp, random_state=RANDOM_STATE, init="pca")
    emb = t.fit_transform(X_sens)
    sil = silhouette_score(emb, y_sens)
    sil_scores.append(sil)
    print(f"perplexity={perp:4d}  silhouette(edge vs non-edge)={sil:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(perplexities, sil_scores, marker="o")
ax.set_xlabel("t-SNE perplexity")
ax.set_ylabel("silhouette score (class separation)")
ax.set_title("t-SNE: cluster separation vs perplexity")
plt.tight_layout()
plt.savefig(FIG_DIR / "q5_perplexity_sensitivity.png", bbox_inches="tight")
plt.show()

## Save metrics

In [ ]:
save_metrics("q5_pca_tsne", {
    "pca_explained_variance_ratio": explained.tolist(),
    "pca_total_variance_explained_2d": float(explained.sum()),
    "tsne_sample_size": int(len(idx)),
    "tsne_perplexity": 30,
    "perplexity_sweep": {"perplexities": perplexities, "silhouette_scores": sil_scores},
})